# Static UMAP Metric Run Script Generator

This notebook is a Delta control panel for the static UMAP metric workflow. It intentionally delegates the real work to the maintained scripts:

1. `static_umap_metric_job_generator.py` creates and submits Slurm job scripts.
2. `static_umap_metrics.py` runs one run/experiment analysis job and writes plots, metric CSVs, summary CSVs, and JSON.

The workflow uses `catalog2.fits` through `catalog-key all`. It does not use separate xmatch catalogs or mmfs paths.

In [1]:
from __future__ import annotations

import shlex
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "static_umap_metric_job_generator.py").exists():
    candidate = NOTEBOOK_DIR / "Hyrax-Research"
    if (candidate / "static_umap_metric_job_generator.py").exists():
        NOTEBOOK_DIR = candidate

GENERATOR = NOTEBOOK_DIR / "static_umap_metric_job_generator.py"
ANALYSIS_SCRIPT = NOTEBOOK_DIR / "static_umap_metrics.py"

if not GENERATOR.exists():
    raise FileNotFoundError(f"Could not find generator script at {GENERATOR}")
if not ANALYSIS_SCRIPT.exists():
    raise FileNotFoundError(f"Could not find analysis script at {ANALYSIS_SCRIPT}")


def run_command(cmd, *, check=True):
    cmd = [str(part) for part in cmd]
    print("$", shlex.join(cmd))
    result = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if result.stdout:
        print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Generator: {GENERATOR}")
print(f"Analysis script: {ANALYSIS_SCRIPT}")

Notebook directory: /work/hdd/bemi/dmiura/Hyrax-Research
Generator: /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metric_job_generator.py
Analysis script: /work/hdd/bemi/dmiura/Hyrax-Research/static_umap_metrics.py


## Configure the batch

The defaults below match the current Delta workflow. Leave `RUN_EXPTS` and `OVERLAY_GROUPS` empty to use the generator's built-in notebook plan: Run 10 gets `time_since_merger` plus `future_merger_flags`, and Run 11 gets `time_since_merger`.

In [3]:
PROFILE = "delta"
BASE_DIR = Path("/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs")
OUTPUT_DIR = BASE_DIR / "static_umap_metrics"

GENERATOR_PYTHON = "python"
JOB_PYTHON = "python"
CATALOG_KEY = "all"
JOB_PREFIX = "plot_metrics"

# Leave empty to use the generator's default Run 10 / Run 11 plan.
# If you add values here, each string should look like "10:7,10,12" or "11:7-14".
RUN_EXPTS = ["10:1-18"] 

# Leave empty to use the generator's per-run defaults. If set, these groups apply to every explicit RUN_EXPTS entry.
OVERLAY_GROUPS = ["time_since_merger", "recent_merger_flags"]
N_PERMUTATIONS = 500
MIN_CLUSTER_SIZE = 15
SEED = 42
DPI = 150
TIME_SINCE_MERGER_MAX_GYR = None
INCLUDE_HIGHDIM = False

DENSITY = False
LOG_COLORBAR = False
SHOW_LEGEND = True
SUPPRESS_LOGS = True

# Example: {"partition": "cpu", "mem": "64G", "time": "4:00:00"}
SLURM_OVERRIDES = {}

# None uses the generator defaults. [] removes setup lines. A list replaces setup lines.
SETUP_LINES = None
NO_DEFAULT_SETUP = False


def build_generator_command(action, *, dry_run=False):
    cmd = [
        GENERATOR_PYTHON,
        GENERATOR,
        action,
        "--profile",
        PROFILE,
        "--base-directory",
        BASE_DIR,
        "--output-dir",
        OUTPUT_DIR,
        "--analysis-script",
        ANALYSIS_SCRIPT,
        "--python-executable",
        JOB_PYTHON,
        "--job-prefix",
        JOB_PREFIX,
        "--catalog-key",
        CATALOG_KEY,
        "--n-permutations",
        N_PERMUTATIONS,
        "--min-cluster-size",
        MIN_CLUSTER_SIZE,
        "--seed",
        SEED,
        "--dpi",
        DPI,
    ]

    for spec in RUN_EXPTS:
        cmd.extend(["--run-expts", spec])
    for group in OVERLAY_GROUPS:
        cmd.extend(["--overlay-group", group])
    for key, value in SLURM_OVERRIDES.items():
        cmd.extend(["--slurm", f"{key}={value}"])

    if TIME_SINCE_MERGER_MAX_GYR is not None:
        cmd.extend(["--time-since-merger-max-gyr", TIME_SINCE_MERGER_MAX_GYR])
    if INCLUDE_HIGHDIM:
        cmd.append("--include-highdim")
    if DENSITY:
        cmd.append("--density")
    if LOG_COLORBAR:
        cmd.append("--log-colorbar")
    if not SHOW_LEGEND:
        cmd.append("--no-show-legend")
    if not SUPPRESS_LOGS:
        cmd.append("--no-suppress-logs")

    if SETUP_LINES is not None:
        if len(SETUP_LINES) == 0:
            cmd.append("--no-default-setup")
        else:
            for line in SETUP_LINES:
                cmd.extend(["--setup-line", line])
    elif NO_DEFAULT_SETUP:
        cmd.append("--no-default-setup")

    if dry_run:
        cmd.append("--dry-run")
    return cmd

print(f"Profile: {PROFILE}")
print(f"Base directory: {BASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Catalog key: {CATALOG_KEY}")

Profile: delta
Base directory: /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs
Output directory: /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/static_umap_metrics
Catalog key: all


## Preview the plan

Run this before creating scripts. It prints the runs, experiments, overlay groups, Slurm settings, catalog key, and output path.

In [ ]:
run_command(build_generator_command("print-plan"))

## Generate Slurm scripts

This writes `plot_metrics<run>_<expt>.sh` into each selected `run<run>/` directory. It does not submit anything.

In [ ]:
run_command(build_generator_command("write"))

## Dry-run submission

This checks the generated scripts and prints the exact `sbatch` commands without submitting jobs.

In [ ]:
run_command(build_generator_command("submit-existing", dry_run=True))

## Submit existing scripts

The guard below prevents accidental submissions. Set `SUBMIT_JOBS = True` only after the dry-run output looks correct.

In [ ]:
SUBMIT_JOBS = False

if SUBMIT_JOBS:
    run_command(build_generator_command("submit-existing"))
else:
    print("SUBMIT_JOBS is False; no jobs submitted.")

## View results

Set `RESULT_RUN` and `RESULT_EXPT` to focus on one result directory, or leave them as `None` to scan every result under `OUTPUT_DIR`. The display helper shows metric CSV rows, overlay-summary rows, PNGs, and recent Slurm log snippets when files exist.

In [ ]:
RESULT_RUN = 10
RESULT_EXPT = None
MAX_TABLE_ROWS = 100
MAX_IMAGES = 0
MAX_LOG_FILES = 0
MAX_LOG_LINES = 0


def result_dirs(run=None, expt=None):
    root = Path(OUTPUT_DIR)
    if run is not None and expt is not None:
        candidates = [root / f"run{int(run)}" / f"expt{int(expt)}"]
    elif run is not None:
        candidates = sorted((root / f"run{int(run)}").glob("expt*"))
    else:
        candidates = sorted(root.glob("run*/expt*"))
    return [path for path in candidates if path.exists()]


def read_result_csvs(pattern, run=None, expt=None):
    frames = []
    for directory in result_dirs(run=run, expt=expt):
        for path in sorted(directory.glob(pattern)):
            frame = pd.read_csv(path)
            frame.insert(0, "source_file", str(path))
            frames.append(frame)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def display_metric_results(run=None, expt=None, max_rows=MAX_TABLE_ROWS, max_images=MAX_IMAGES):
    directories = result_dirs(run=run, expt=expt)
    print(f"Output root: {Path(OUTPUT_DIR)}")
    print(f"Result directories found: {len(directories)}")
    if not directories:
        print("No result directories found yet.")
        return

    metrics = read_result_csvs("*_metrics.csv", run=run, expt=expt)
    if metrics.empty:
        print("No metrics CSV files found yet.")
    else:
        display(Markdown("### Metrics CSV summary"))
        display(metrics.head(max_rows))

    overlay_summary = read_result_csvs("*_overlay_summary.csv", run=run, expt=expt)
    if overlay_summary.empty:
        print("No overlay-summary CSV files found yet.")
    else:
        display(Markdown("### Overlay summary CSV"))
        display(overlay_summary.head(max_rows))

    pngs = []
    for directory in directories:
        pngs.extend(sorted(directory.glob("*.png")))

    if not pngs:
        print("No PNG plots found yet.")
        return

    display(Markdown(f"### PNG plots ({min(len(pngs), max_images)} of {len(pngs)})"))
    for path in pngs[:max_images]:
        try:
            label = path.relative_to(OUTPUT_DIR)
        except ValueError:
            label = path
        display(Markdown(f"**{label}**"))
        display(Image(filename=str(path)))


def display_log_snippets(run=None, expt=None, max_files=MAX_LOG_FILES, max_lines=MAX_LOG_LINES):
    base = Path(BASE_DIR)
    if run is not None:
        run_dirs = [base / f"run{int(run)}"]
    else:
        run_dirs = sorted(base.glob("run*"))

    files = []
    for run_dir in run_dirs:
        if not run_dir.exists():
            continue
        if run is not None and expt is not None:
            pattern = f"{JOB_PREFIX}{int(run)}_{int(expt)}.txt"
        else:
            pattern = f"{JOB_PREFIX}*.txt"
        files.extend(sorted(run_dir.glob(pattern)))

    if not files:
        print("No Slurm log files found yet.")
        return

    display(Markdown(f"### Slurm log snippets ({min(len(files), max_files)} of {len(files)})"))
    for path in files[:max_files]:
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
        tail = "\n".join(lines[-max_lines:])
        display(Markdown(f"**{path}**"))
        print(tail if tail else "[empty log]")

In [ ]:
display_metric_results(run=RESULT_RUN, expt=RESULT_EXPT)
display_log_snippets(run=RESULT_RUN, expt=RESULT_EXPT)